# Structured Output

When we interact with an LLM directly, it typically responds with plain text.  
This is perfect for use cases like chatbots, RAG systems, or assistants where we want natural-language responses.

But sometimes we need the output in a **specific structure** — for example:

- JSON for API responses  
- Objects for further processing  
- CSV-like rows  
- Typed schemas (Pydantic models, dataclasses, etc.)

This is where **structured output** becomes useful. We can instruct the LLM to return data in a fixed shape instead of free-form text.

Not all models natively support structured output. Some do (e.g., OpenAI’s modern models, Anthropic’s Claude), and some don’t. Always check your model’s documentation if you plan to rely on the model’s built-in structured output abilities.

However, LangChain abstracts all this away and (seemingly) gives *any* model the ability to produce structured output.


### Response Strategies in LangChain

LangChain uses different strategies depending on the model’s abilities and the schema you provide:

### **1. `ProviderStrategy[T]`**
Uses the model's **native structured output** capabilities.  
This is the cleanest and most reliable approach — but only works if the model explicitly supports structured outputs.

### **2. `ToolStrategy[T]`**
For models that *don’t* support structured outputs.

LangChain automatically falls back to **tool calling** to simulate structured output:
- It wraps your schema into a virtual tool
- The model “calls” it with the structured arguments
- LangChain extracts those arguments as the structured output

This makes *any model* capable of structured output.

### **3. `type[T]`**
Schema is provided directly:
- It inspects the model’s capabilities
- Automatically picks the best strategy (`ProviderStrategy` or `ToolStrategy`)

You usually don’t need to manually choose — LangChain will handle it for you.


Let's look at some examples

> ⚠️ **Environment Variables**  
> This notebook requires an <code>OPENAI_API_KEY</code> to run.  
> You can also modify the code if you want to use a different model provider.

In [63]:
from langchain.agents import create_agent
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import HumanMessage, SystemMessage
from pprint import pprint
import json

load_dotenv()

True

### Example

To demonstrate the usage of structured output, let's extract a data structure from a natural language description of a movie scene.

We will ask the model to analyze:
- Genre
- Main characters
- Emotional tone
- A one-sentence summary
- Who the movie is best suited for

### Define the Schema
LangChain supports multiple ways of defining sctructured output:
- Pydantic models
- Dataclasses
- TypedDicts
- JSON Schema

I like working with pydantic classes, so we will use them in this example

In [42]:
class Character(BaseModel):
    name: str = Field(..., description="Name of the character")
    role: str = Field(..., description="The character's role in the movie")

class MovieAnalysis(BaseModel):
    title: str = Field(..., description="Movie title")
    genre: str = Field(..., description="Primary genre of the movie")
    summary: str = Field(..., description="One-sentence summary of the movie")
    emotional_tone: str = Field(..., description="Overall emotional tone")
    main_characters: List[Character] = Field(..., description="List of important characters")
    recommended_audience: str = Field(..., description="Who would most enjoy the movie")

Create a system prompt.

In [94]:
system_prompt = """
    You are a professional movie analyst. Your task is to extract structured information from a movie description. 
    Always return a JSON object that matches the provided schema exactly. **IMPORTANT**: Do not add any commentary or free text.
"""

Prepare example movie description

In [44]:
movie_description = """
    A retired hitman named Marcus is forced back into action when a cyber-crime syndicate 
    kidnaps his daughter. He teams up with an idealistic young hacker to dismantle the 
    organization. The film mixes neon-soaked cityscapes, fast-paced fights, and emotional 
    moments about loyalty and redemption.
"""

### Provider Strategy
If the model supports structured output natively, this is the best strategy to choose. The model provider enforces the schema and thus provides high realibility and strict validation.

Here we are using `gpt-4o-mini` which has the capability to return structured output, so we will use `ProviderStrategy`

In [99]:
provider_strategy_agent = create_agent(
    model='gpt-4o-mini',
    response_format=ProviderStrategy(MovieAnalysis),
)

Invoke the model

In [ ]:
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=movie_description)
]

provider_strategy_result = provider_strategy_agent.invoke({
    'messages': messages
})

In [ ]:
provider_strategy_json = provider_strategy_result['messages'][-1].content

# parse for pretty print
data = json.loads(provider_strategy_json)
print(json.dumps(data, indent=2))

{
  "title": "Unforgiven Code",
  "genre": "Action/Thriller",
  "summary": "A retired hitman named Marcus teams up with a young hacker to rescue his kidnapped daughter from a cyber-crime syndicate.",
  "emotional_tone": "Intense and Emotional",
  "main_characters": [
    {
      "name": "Marcus",
      "role": "Retired hitman"
    },
    {
      "name": "Young Hacker",
      "role": "Idealistic ally"
    }
  ],
  "recommended_audience": "Fans of action-packed thrillers and cybercrime stories."
}


### Tool Strategy
To demonstrate the usage of `ToolStrategy` we will use `gpt-3.5-turbo` that does not natively support structured output. If we try to invoke it with `ProviderStrategy` it will throw an error.

In [123]:
tool_strategy_agent = create_agent(
    model='gpt-3.5-turbo',
    response_format=ToolStrategy(
        schema=MovieAnalysis,
        tool_message_content=""
    ),
)

In [124]:
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=movie_description)
]

tool_strategy_result = tool_strategy_agent.invoke({
    'messages': messages
})

The structured output can be found uder `structured_response`

In [ ]:
print(json.dumps(tool_strategy_result['structured_response'].model_dump(), indent=4))

{
    "title": "Redemption",
    "genre": "Action",
    "summary": "A retired hitman named Marcus is forced back into action when a cyber-crime syndicate kidnaps his daughter. He teams up with an idealistic young hacker to dismantle the organization. The film mixes neon-soaked cityscapes, fast-paced fights, and emotional moments about loyalty and redemption.",
    "emotional_tone": "Intense",
    "main_characters": [
        {
            "name": "Marcus",
            "role": "Retired Hitman"
        },
        {
            "name": "Hacker",
            "role": "Idealistic Young Hacker"
        }
    ],
    "recommended_audience": "Action movie fans"
}
